In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# from torch_geometric.utils.num_nodes import maybe_num_nodes
# from torch_geometric.utils._subgraph import subgraph
import torch_geometric.nn as geom_nn
import os
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_score,accuracy_score,recall_score,f1_score,roc_auc_score
from wsi.nets.dasmil import DASMIL


In [ ]:
model = DASMIL().cuda()
model.eval()
state_dict_weights = torch.load("./model/base/DAS/dsamil.pt", map_location=torch.device('cpu'))
model.load_state_dict(state_dict_weights, strict=False)



In [ ]:
data = torch.load("../../camGraph_23/processed/test/data_10.pt", weights_only=False).cuda()
x, edge_index, childof, level, edge_index2, edge_index3 = data.x, data.edge_index, data.childof, data.level, data.edge_index_2, data.edge_index_3
data.x_coord[data.level==2],data.y_coord[data.level==2]
len(data.x[data.level==2]),len(data.x[data.level==3])
edge_index2.shape
# results = model(x, edge_index, level, childof,edge_index2, edge_index3)


In [ ]:
path = "../../camGraph_23/processed/test/"
f_list = [os.path.join(path,f) for f in os.listdir(path)]
result_list = []
for f in f_list:
    data = torch.load(f, weights_only=False).cuda()
    x, edge_index, childof, level, edge_index2, edge_index3 = data.x, data.edge_index, data.childof, data.level, data.edge_index_2, data.edge_index_3

    results = model(x, edge_index, level, childof,edge_index2, edge_index3)
    result_list.append([model.predict(results)[0].item(),data.y.item()])



In [ ]:
results["higher"][0].shape,results["higher"][1].shape,results["higher"][2].shape,results["higher"][3].shape

In [ ]:
pred = np.array(result_list)[:,0]
label = np.array(result_list)[:,1]
pred[pred>0.5]=1
pred[pred<1]=0

precision_score(label,pred), accuracy_score(label,pred), recall_score(label,pred), f1_score(label,pred), roc_auc_score(label,pred)